# MuonClip + RMS: layer ESDs and `fix_fingers='clip_xmax'`

This notebook inspects the six hidden matrices from a one-head nanoGPT epoch checkpoint using the same CPU-only `WeightMatrixHolder` used by the training diagnostics. It does **not** touch or pause the live training process.

It does four things:

1. loads the latest completed quarter-epoch model checkpoint (or a selected `TARGET_EPOCH`);
2. reproduces the standard WeightWatcher analysis with `ERG=True`, `randomize=True`, and `min_evals=20`;
3. reruns the same six matrices with `fix_fingers='clip_xmax'` and configurable `MAX_FINGERS`;
4. compares `alpha`, `D`, `xmin`, `xmax`, `raw_alpha`, and `num_fingers`, then plots all six empirical spectral densities on a compact log-log grid.

The purpose is to test whether unusually large fitted alphas are dominated by a small number of finite-size fingers at the top of the spectrum rather than by the shape of the full heavy-tailed ESD.

In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
import weightwatcher as ww

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'baseline' / 'nanogpt_one_head']
EXPERIMENT_ROOT = next(
    (path for path in candidates if (path / 'configs' / 'reference.yaml').is_file()),
    None,
)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError('Run from baseline/nanogpt_one_head or the repository root')
sys.path.insert(0, str(EXPERIMENT_ROOT / 'src'))

from rg_nanogpt_one_head.model import GPT, GPTConfig
from rg_nanogpt_one_head.spectral import WeightMatrixHolder, _attach_matrix_metadata

# Change RUN_DIR to inspect another baseline run.
RUN_DIR = Path('/tmp/rg-nanogpt-long-muonclip-50ep/results/muon_clip/seed_1337')

# None means: use the latest completed WeightWatcher/epoch checkpoint.
# Example: TARGET_EPOCH = 4.0
TARGET_EPOCH = None

MIN_EVALS = 20
MAX_FINGERS = 10
RANDOMIZE = True
ERG = True

print('experiment root:', EXPERIMENT_ROOT)
print('run dir:', RUN_DIR)
print('WeightWatcher version:', getattr(ww, '__version__', 'unknown'))

## Load the exact epoch checkpoint used for a WeightWatcher measurement

Quarter-epoch checkpoints are model-only CPU-portable snapshots. Selecting from `epoch_metrics.csv` keeps this analysis aligned with the stored `spectral/layers.csv` row at the same optimizer step.

In [ ]:
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f'Run directory does not exist: {RUN_DIR}')

manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
epoch_metrics = pd.read_csv(RUN_DIR / 'epoch_metrics.csv')
for column in ['step', 'nominal_epoch', 'epoch']:
    epoch_metrics[column] = pd.to_numeric(epoch_metrics[column], errors='coerce')
epoch_metrics = epoch_metrics.dropna(subset=['step', 'nominal_epoch']).sort_values('step')

if TARGET_EPOCH is None:
    selected = epoch_metrics.iloc[-1]
else:
    idx = (epoch_metrics['nominal_epoch'] - float(TARGET_EPOCH)).abs().idxmin()
    selected = epoch_metrics.loc[idx]

STEP = int(selected['step'])
NOMINAL_EPOCH = float(selected['nominal_epoch'])
ACTUAL_EPOCH = float(selected['epoch'])

checkpoint_path = Path(str(selected.get('checkpoint_path', '')))
if not checkpoint_path.is_file():
    matches = sorted((RUN_DIR / 'epoch_checkpoints').glob(f'*step_{STEP:07d}.pt'))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Could not resolve the model checkpoint for step {STEP}; found {matches}'
        )
    checkpoint_path = matches[0]

payload = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model = GPT(GPTConfig(**manifest['model']))
model.load_state_dict(payload['model'])
model.eval()
holder = WeightMatrixHolder(model)

OUTPUT_DIR = RUN_DIR / 'diagnostics' / f'esd_clip_xmax_step_{STEP:07d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('checkpoint:', checkpoint_path)
print(f'step={STEP} nominal_epoch={NOMINAL_EPOCH:.3f} actual_epoch={ACTUAL_EPOCH:.6f}')
print('output dir:', OUTPUT_DIR)
print('matrices:', [item['matrix_name'] for item in holder.matrix_metadata])

stored_path = RUN_DIR / 'spectral' / 'raw' / f'weightwatcher_step_{STEP:07d}.csv'
if stored_path.is_file():
    stored = pd.read_csv(stored_path)
    cols = [c for c in ['matrix_name', 'alpha', 'D', 'rand_distance', 'ERG_gap', 'num_traps'] if c in stored.columns]
    print('\nStored training-time WeightWatcher result at this step:')
    display(stored[cols].sort_values('matrix_name'))


## Standard WeightWatcher analysis

This is the same analysis contract used during training: `ERG=True`, `randomize=True`, `plot=True`, `min_evals=20`. WeightWatcher will emit its native ESD / fit figures for all six matrices.

In [ ]:
diagnostic_seed = int(manifest['seed']) + 1_000_003 + STEP

def reset_diagnostic_seed():
    random.seed(diagnostic_seed)
    np.random.seed(diagnostic_seed % (2**32 - 1))
    torch.manual_seed(diagnostic_seed)

def attach(details, holder):
    return _attach_matrix_metadata(pd.DataFrame(details), holder.matrix_metadata)

def show_details(frame):
    preferred = [
        'matrix_name', 'alpha', 'raw_alpha', 'D', 'xmin', 'xmax',
        'num_fingers', 'num_pl_spikes', 'ERG_gap', 'rand_distance', 'warning'
    ]
    cols = [column for column in preferred if column in frame.columns]
    display(frame[cols].sort_values('matrix_name'))

reset_diagnostic_seed()
watcher_standard = ww.WeightWatcher(model=holder)
details_standard_raw = watcher_standard.analyze(
    ERG=ERG,
    randomize=RANDOMIZE,
    plot=True,
    min_evals=MIN_EVALS,
)
details_standard = attach(details_standard_raw, holder)
details_standard.to_csv(OUTPUT_DIR / 'weightwatcher_standard.csv', index=False)
show_details(details_standard)

## Rerun with `fix_fingers='clip_xmax'`

The second pass keeps the same model, diagnostic seed, ERG/randomization settings, and minimum ESD size. The only fit change is `fix_fingers='clip_xmax'` with `max_fingers=MAX_FINGERS`. No fallback is used: if the installed WeightWatcher build rejects this option, the cell raises the underlying error so the result cannot be mistaken for a successful finger correction.

In [ ]:
reset_diagnostic_seed()
watcher_clip = ww.WeightWatcher(model=holder)
details_clip_raw = watcher_clip.analyze(
    ERG=ERG,
    randomize=RANDOMIZE,
    plot=True,
    min_evals=MIN_EVALS,
    fix_fingers='clip_xmax',
    max_fingers=MAX_FINGERS,
)
details_clip = attach(details_clip_raw, holder)
details_clip.to_csv(OUTPUT_DIR / 'weightwatcher_clip_xmax.csv', index=False)
show_details(details_clip)

## Standard versus `clip_xmax` fit comparison

A large drop in `alpha` accompanied by a nonzero `num_fingers` is the signature we are looking for: it says the standard fit was being driven by a small number of top-of-spectrum finite-size eigenvalues. If `alpha` stays high after clipping, the high exponent is not explained by that mechanism.

In [ ]:
metrics_to_compare = ['alpha', 'raw_alpha', 'D', 'xmin', 'xmax', 'num_fingers', 'num_pl_spikes']
std_cols = ['matrix_name'] + [c for c in metrics_to_compare if c in details_standard.columns]
clip_cols = ['matrix_name'] + [c for c in metrics_to_compare if c in details_clip.columns]

comparison = (
    details_standard[std_cols]
    .rename(columns={c: f'{c}_standard' for c in std_cols if c != 'matrix_name'})
    .merge(
        details_clip[clip_cols].rename(columns={c: f'{c}_clip_xmax' for c in clip_cols if c != 'matrix_name'}),
        on='matrix_name',
        how='inner',
    )
)
if {'alpha_standard', 'alpha_clip_xmax'} <= set(comparison.columns):
    comparison['delta_alpha_clip_minus_standard'] = (
        comparison['alpha_clip_xmax'] - comparison['alpha_standard']
    )
    comparison['alpha_reduction'] = (
        comparison['alpha_standard'] - comparison['alpha_clip_xmax']
    )

comparison = comparison.sort_values('matrix_name')
comparison.to_csv(OUTPUT_DIR / 'standard_vs_clip_xmax.csv', index=False)
display(comparison)

if 'alpha_standard' in comparison and 'alpha_clip_xmax' in comparison:
    print('median alpha, standard :', float(comparison['alpha_standard'].median()))
    print('median alpha, clip_xmax:', float(comparison['alpha_clip_xmax'].median()))
    print('mean alpha, standard   :', float(comparison['alpha_standard'].mean()))
    print('mean alpha, clip_xmax  :', float(comparison['alpha_clip_xmax'].mean()))

## Compact six-layer ESD grid

WeightWatcher already produced its native diagnostic figures above. This cell uses `watcher.get_ESD(...)` to put all six empirical ESDs on one page and overlays the fit boundaries from the standard and `clip_xmax` analyses. The ESD itself is unchanged; `clip_xmax` changes the fitted tail selection.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
axes = axes.ravel()

for ax, (_, row) in zip(axes, details_standard.sort_values('matrix_name').iterrows()):
    matrix_name = str(row['matrix_name'])
    layer_id = int(row['layer_id'])
    esd = np.asarray(watcher_standard.get_ESD(layer=layer_id), dtype=float)
    esd = esd[np.isfinite(esd) & (esd > 0)]

    if esd.size == 0:
        ax.set_title(f'{matrix_name}: no positive eigenvalues')
        continue

    lo = float(esd.min())
    hi = float(esd.max())
    if hi <= lo:
        bins = np.linspace(lo * 0.99, hi * 1.01 + 1e-12, 16)
    else:
        bins = np.logspace(np.log10(lo), np.log10(hi), 32)
    hist, edges = np.histogram(esd, bins=bins, density=True)
    centers = np.sqrt(edges[:-1] * edges[1:])
    valid = np.isfinite(hist) & (hist > 0) & np.isfinite(centers) & (centers > 0)
    ax.loglog(centers[valid], hist[valid], marker='o', linestyle='none', label='empirical ESD')

    clip_row = details_clip.loc[details_clip['matrix_name'] == matrix_name].iloc[0]
    for value, label, style in [
        (row.get('xmin', np.nan), 'xmin standard', '--'),
        (clip_row.get('xmin', np.nan), 'xmin clip_xmax', ':'),
        (clip_row.get('xmax', np.nan), 'xmax clip_xmax', '-.'),
    ]:
        value = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
        if np.isfinite(value) and value > 0:
            ax.axvline(float(value), linestyle=style, label=label)

    alpha_std = float(row['alpha']) if pd.notna(row.get('alpha')) else float('nan')
    alpha_clip = float(clip_row['alpha']) if pd.notna(clip_row.get('alpha')) else float('nan')
    ax.set_title(f'{matrix_name}  alpha: {alpha_std:.3f} -> {alpha_clip:.3f}')
    ax.set_xlabel('eigenvalue of X = W^T W')
    ax.set_ylabel('density')
    ax.grid(True, which='both', alpha=0.2)
    ax.legend(fontsize=8)

fig.suptitle(
    f'MuonClip + RMS, epoch {NOMINAL_EPOCH:.2f}, step {STEP}: ESD and clip_xmax fit boundaries',
    fontsize=14,
)
figure_path = OUTPUT_DIR / 'all_layer_esds_standard_vs_clip_xmax.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', figure_path)

## What to inspect

For the current high-alpha MuonClip run, start with `W_V` and `W_O`. Compare the native standard and `clip_xmax` WeightWatcher plots and the table above.

- **Large alpha reduction + nonzero fingers:** the original alpha was likely dominated by the uppermost finite-size eigenvalues.
- **Little alpha reduction:** the high alpha is not fixed by clipping the upper finger eigenvalues; the layer may still be undertrained / random-like or may simply not have a clean single-PL tail yet.
- **Watch `D` as well as alpha:** a smaller alpha is not useful if the corrected fit quality becomes poor.
- **Use the native ESD plots:** visually inspect whether the fitted tail spans a meaningful fraction of the spectrum rather than only a few points.

All CSV outputs and the compact ESD figure are written under `RUN_DIR/diagnostics/esd_clip_xmax_step_XXXXXXX/`, so the same notebook can be rerun at later epochs without overwriting earlier results.